# recs_022 — V2b IGDB summary USE cosine spike

**Summary similarity rerank on frozen `two_tower_v1` @100 pools · train_tune → val**

V2b signal: **USE dot product** between the user's **query review** embedding and each pool candidate's precomputed **`summary__use`** (IGDB official pitch text). Same USE encoder as v1 retrieval.

**Plan:** [`docs/recommender_v2_plan.md`](../../docs/recommender_v2_plan.md) · **Shipped v2a:** [`recs_020_v2a_taxonomy_use_cosine.ipynb`](recs_020_v2a_taxonomy_use_cosine.ipynb)

# Executive Summary

**Question:**  
Does IGDB **summary** similarity (`sim(query_review, igdb_summary)`) improve reranking vs D1 and vs shipped **v2a**?

**Result:**  
Pure summary rerank (`two_tower_v1_v2b_igdb_summary`, α=0) **fails** vs D1: val **0.032** / **0.036** overall / Slice A. **`two_tower_v1_v2b_igdb_summary_logpop_blend`** (D1 + summary, `w_summary=0.1`) **beats D1** on val: overall NDCG@10 **0.093** / Slice A **0.072** vs D1 **0.093** / **0.068** — but **loses overall** to shipped v2a (**0.095** / **0.070**). Slice A is marginally ahead of v2a (**0.072** vs **0.070**).

**Recommendation / Decision:**  
**Kill** summary-only rerank for ship. **`_logpop_blend`** is a **defer** candidate — beats D1 but does not clear the **shipped v2a** bar on overall NDCG. Head-to-head vs v2a before any eval-job wiring.

# Business Context

V2a (taxonomy metadata) shipped as `two_tower_v1_v2a_embed_query_logpop_blend`. V2b tests whether **publisher/editorial copy** (IGDB summary) adds ordering signal *within* frozen pools — orthogonal to review-profile retrieval and taxonomy tags.

**Expected impact:** If summary sim lifts Slice A without hurting personalization, it becomes a V2c blend ingredient.

# Research Question

On `train_ranker_v1` / `val_dev_12k_v1`, does `cosine(embed(query_review), summary__use(candidate))` rerank frozen pools better than D1 and shipped v2a?

# Hypothesis

**Hypothesis:**  
Official game summaries capture pitch-level semantics (genre, setting, mechanics) that complement D1's retrieval+log-pop blend but were not fully used by taxonomy-only v2a.

**Success criteria:** Beat **shipped v2a** on val NDCG@10 overall **and** Slice A without worsening personalization vs v2a.

# Definitions

| Term | Definition |
|------|------------|
| V2b | Summary USE cosine rerank (this spike) |
| `query_review` | `query_text` from eval cohort — the user's review at query time |
| `summary__use` | 512-d L2-normalized USE embedding of IGDB `summary` (Job 1) |
| α | Retrieval blend weight in `α·retr + (1−α)·summary` |
| `w_summary` | D1+summary blend weight: `(1−w)·norm(D1) + w·norm(summary)` |
| D1 | `two_tower_v1_heuristic_logpop_blend` (`α=0.2` retr + log-pop) |

# Data Sources

| Source | Path |
|--------|------|
| Train pools | `artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet` |
| Train cohort (`query_text`) | `artifacts/recs/eval_cache/train_ranker_v1/example_cohort.parquet` |
| Val pools | `artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl` |
| Val examples | `artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet` |
| IGDB summaries + USE | `artifacts/igdb/lookups/games.parquet` (`summary`, `summary__use`) |

Prerequisites: IGDB Job 1 (`recs_job_igdb_games.py`), train pool export, offline eval jsonl.

# Design / Process

1. Load `summary__use` per catalog `app_id` from IGDB games lookup
2. Embed `query_text` per example with **same USE** as v1 (`ContentRetriever.embed_text`)
3. Score pool: cosine(query_review, candidate `summary__use`); min-max normalize within pool
4. Grid-search α (retr+summary) on **train_tune** (Slice A NDCG@10)
5. Grid-search `w_summary` (D1+summary `_logpop_blend`) on train_tune
6. Val face-off vs D1, bare retrieval, shipped v2a, oracle

```bash
python scripts/recs_job_igdb_games.py configs/recs_job_igdb_games.json
python scripts/recs_job_export_retrieval_pools.py configs/recs_job_export_retrieval_pools_train_ranker.json
python scripts/recs_job_eval_offline.py configs/recs_job_eval_offline.json \
  --examples-parquet artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet
```

# Decision Log

| Decision | Reason |
|----------|--------|
| Use Job 1 `summary__use` (not enriched parquet) | Precomputed at IGDB pull; 100% catalog coverage |
| Embed `query_text` on the fly | Per-example review text; not keyed by `query_app_id` |
| No history anchor | V2b contract is query-review ↔ candidate summary only |
| Tune on train_tune only | Same 90/10 stratified split as v2a spikes |

# Evaluation Outputs / Artifacts

| Artifact | Path |
|----------|------|
| Train grid (retr+summary α) | `artifacts/recs/spikes/v2b/v2b_train_tune_grid.csv` |
| Train grid (D1+summary `w_summary`) | `artifacts/recs/spikes/v2b/v2b_plus_d1_train_tune_grid.csv` |
| Val per-example | `artifacts/recs/spikes/v2b/v2b_val_per_example.parquet` |
| Val overall / slice | `artifacts/recs/spikes/v2b/v2b_val_*.csv` |

# Notebook Roadmap

1. Setup and paths
2. Load pools, catalog, IGDB `summary__use`, query texts
3. Validate summary coverage
4. Summary scoring helpers + train_tune grids
5. Val face-off + metric tables
6. Findings

# Analysis

## Setup

In [ ]:
from __future__ import annotations

import json
from pathlib import Path
from typing import Any, Callable

import numpy as np
import pandas as pd
from IPython.display import Markdown, display

from steam_review_ml.evaluation.example_cohort import load_retrieval_pool_rows, load_retrieval_pools_jsonl
from steam_review_ml.evaluation.heuristic_ranker import (
    DEFAULT_LOGPOP_BLEND_ALPHA,
    minmax_norm,
    score_logpop_blend,
)
from steam_review_ml.evaluation.retrieval_offline_eval import (
    RANKING_REPORT_METRIC_COLS,
    _append_personalization_metrics,
    _oracle_ranked_indices_from_retrieved,
    _rank_rows,
    _table_by_slice_for_metrics,
    _table_by_support_for_metrics,
    _table_overall_ranking,
    _table_personalization,
    _table_popularity,
    average_precision_at_k,
    hit_rate_at_k,
    load_ranking_catalog_context,
    mrr,
    ndcg_at_k,
    precision_at_k,
    recall_at_k,
)
from steam_review_ml.evaluation.v2a_metadata_ranker import (
    DEFAULT_V2A_EMBED_W_META,
    V2A_GENRE_THEME_KW_FIELDS,
    score_v2a_embed_query_logpop_blend,
)
from steam_review_ml.igdb.constants import USE_EMBEDDING_FIELD_SUFFIX
from steam_review_ml.recommender.retrieve import ContentRetriever

REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
SPLIT_SEED = 42
TUNE_FRAC = 0.10
K_FINAL = 10
K_PERSONALIZATION = 10
MIN_REVIEW_CHARS = 30
POOL_METHOD = "two_tower_v1"
D1_ALPHA = DEFAULT_LOGPOP_BLEND_ALPHA

TRAIN_POOLS_PARQUET = REPO_ROOT / "artifacts/recs/ranker_pools/train_ranker_v1/two_tower_v1.parquet"
TRAIN_COHORT_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/train_ranker_v1/example_cohort.parquet"
VAL_JSONL = REPO_ROOT / "artifacts/recs/offline_eval/runs/latest/eval_offline_examples.jsonl"
VAL_EXAMPLES_PARQUET = REPO_ROOT / "artifacts/recs/eval_cache/val_dev_12k_v1/eval_examples.parquet"
IGDB_GAMES = REPO_ROOT / "artifacts/igdb/lookups/games.parquet"
ARTIFACT_DIR = REPO_ROOT / "artifacts/recs"
SPIKE_OUT = ARTIFACT_DIR / "spikes/v2b"
SUMMARY_USE_COL = f"summary{USE_EMBEDDING_FIELD_SUFFIX}"

BLEND_ALPHAS = [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]
PLUS_D1_WEIGHTS = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3]

for p in (TRAIN_POOLS_PARQUET, TRAIN_COHORT_PARQUET, VAL_JSONL, VAL_EXAMPLES_PARQUET, IGDB_GAMES):
    if not p.is_file():
        raise FileNotFoundError(f"Missing required artifact: {p}")

SPIKE_OUT.mkdir(parents=True, exist_ok=True)
print(f"REPO_ROOT={REPO_ROOT}")
print(f"SPIKE_OUT={SPIKE_OUT}")

## Load Data

In [ ]:
train_pools = load_retrieval_pool_rows(TRAIN_POOLS_PARQUET)
train_cohort = pd.read_parquet(TRAIN_COHORT_PARQUET, columns=["ex_idx", "query_text"])
val_pools = load_retrieval_pools_jsonl(VAL_JSONL, method=POOL_METHOD)
val_pools_by_ex = {int(r["ex_idx"]): r for r in val_pools}

catalog = load_ranking_catalog_context(
    repo_root=REPO_ROOT, min_review_chars=MIN_REVIEW_CHARS, artifact_dir=ARTIFACT_DIR,
)
app_ids = catalog.app_ids
app_to_row = catalog.app_to_row
pop_row = catalog.pop_row

retriever = ContentRetriever(artifact_dir=ARTIFACT_DIR, repo_root=REPO_ROOT)
X_emb = retriever.embedding_matrix

val_examples_df = pd.read_parquet(VAL_EXAMPLES_PARQUET)
examples_for_pers: list[dict[str, Any]] = []
for _, row in val_examples_df.iterrows():
    examples_for_pers.append(
        {
            "ex_idx": int(row["ex_idx"]),
            "user_id": row["user_id"],
            "query_app_id": int(row["query_app_id"]),
            "query_ts": float(row["query_ts"]),
            "n_eval_targets": int(row["n_eval_targets"]),
            "train_review_rows": json.loads(row["train_review_rows_json"]),
            "validation_positive_app_ids": json.loads(row["validation_positive_app_ids_json"]),
        }
    )

support_by_ex = val_examples_df.set_index("ex_idx")["n_support_train"].astype(int).to_dict()
for row in val_pools:
    row["n_support_train"] = int(support_by_ex.get(int(row["ex_idx"]), 0))

print(
    f"train={len(train_pools):,} val={len(val_pools):,} catalog={len(app_ids):,}"
)

## Validate Data Quality

In [ ]:
def parse_summary_use(val: Any) -> np.ndarray:
    """Normalize ``summary__use`` to 512-d (zeros if missing)."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return np.zeros(512, dtype=np.float64)
    arr = np.asarray(val, dtype=np.float64)
    return arr.reshape(512) if arr.size == 512 else np.zeros(512, dtype=np.float64)


igdb_df = pd.read_parquet(IGDB_GAMES, columns=["app_id", "summary", SUMMARY_USE_COL])
summary_by_app: dict[int, np.ndarray] = {
    int(row["app_id"]): parse_summary_use(row[SUMMARY_USE_COL]) for _, row in igdb_df.iterrows()
}

catalog_app_set = {int(a) for a in app_ids}
nonzero_summary = sum(
    1 for a in catalog_app_set if np.linalg.norm(summary_by_app.get(a, 0)) > 1e-8
)
display(pd.Series({"catalog_apps_with_summary_use": nonzero_summary, "catalog_total": len(catalog_app_set)}))

# Pool-level coverage: share of pool slots with nonzero summary embedding
pool_slots = 0
covered_slots = 0
for row in val_pools[:500]:  # sample for speed
    for app_id in json.loads(row["retrieved_app_ids_json"]):
        pool_slots += 1
        if np.linalg.norm(summary_by_app.get(int(app_id), 0)) > 1e-8:
            covered_slots += 1
print(f"sample pool summary coverage: {covered_slots / pool_slots:.1%} ({covered_slots:,}/{pool_slots:,} slots)")

## Feature Engineering / Preprocessing

In [ ]:
def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    na, nb = float(np.linalg.norm(a)), float(np.linalg.norm(b))
    if na <= 1e-12 or nb <= 1e-12:
        return 0.0
    return float(np.dot(a, b) / (na * nb))


def build_query_vec_map(cohort_df: pd.DataFrame, retriever: ContentRetriever) -> dict[int, np.ndarray]:
    """Embed unique query texts once; map ex_idx → 512-d review vector."""
    texts = cohort_df.set_index("ex_idx")["query_text"].astype(str).to_dict()
    unique_texts = sorted(set(texts.values()))
    text_to_vec = {t: np.asarray(retriever.embed_text(t), dtype=np.float64) for t in unique_texts}
    return {ex_idx: text_to_vec[str(text)] for ex_idx, text in texts.items()}


train_query_vecs = build_query_vec_map(train_cohort, retriever)
val_query_vecs = build_query_vec_map(val_examples_df[["ex_idx", "query_text"]], retriever)
print(f"embedded train={len(train_query_vecs):,} val={len(val_query_vecs):,} unique texts")

In [ ]:
def raw_summary_pool_scores(
    pool_app_ids: list[int], *, query_vec: np.ndarray, summary_by_app: dict[int, np.ndarray]
) -> np.ndarray:
    out = np.empty(len(pool_app_ids), dtype=np.float64)
    for i, app_id in enumerate(pool_app_ids):
        out[i] = cosine_sim(query_vec, summary_by_app.get(int(app_id), np.zeros(512)))
    return out


def summary_blend_pool_scores(
    pool_app_ids: list[int],
    retrieval_scores: list[float] | np.ndarray,
    *,
    alpha: float,
    query_vec: np.ndarray,
    summary_by_app: dict[int, np.ndarray],
) -> np.ndarray:
    meta = raw_summary_pool_scores(pool_app_ids, query_vec=query_vec, summary_by_app=summary_by_app)
    retr = minmax_norm(np.asarray(retrieval_scores, dtype=np.float64))
    meta_n = minmax_norm(meta)
    return alpha * retr + (1.0 - alpha) * meta_n


def d1_plus_summary_pool_scores(
    pool_app_ids: list[int],
    retrieval_scores: list[float] | np.ndarray,
    *,
    w_summary: float,
    query_vec: np.ndarray,
    summary_by_app: dict[int, np.ndarray],
    pop_row: np.ndarray,
    app_to_row: dict[int, int],
) -> np.ndarray:
    d1 = score_logpop_blend(
        pool_app_ids, retrieval_scores, alpha=D1_ALPHA, pop_row=pop_row, app_to_row=app_to_row
    )
    meta = raw_summary_pool_scores(pool_app_ids, query_vec=query_vec, summary_by_app=summary_by_app)
    if w_summary <= 0.0:
        return d1
    if w_summary >= 1.0:
        return minmax_norm(meta)
    return (1.0 - w_summary) * minmax_norm(d1) + w_summary * minmax_norm(meta)


def pool_scores_to_ranked_indices(pool_app_ids: list[int], pool_scores: np.ndarray, *, k_final: int) -> np.ndarray:
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, score in zip(pool_app_ids, pool_scores):
        full[int(app_to_row[int(app_id)])] = float(score)
    return _rank_rows(full)[:k_final]


def stratified_ex_idx_split(pools: list[dict[str, Any]], *, tune_frac: float, seed: int) -> tuple[set[int], set[int]]:
    rng = np.random.default_rng(seed)
    by_slice: dict[str, list[int]] = {}
    for row in pools:
        by_slice.setdefault(str(row["slice_name"]), []).append(int(row["ex_idx"]))
    fit_ids: set[int] = set()
    tune_ids: set[int] = set()
    for ids in by_slice.values():
        ids_arr = np.asarray(sorted(ids))
        rng.shuffle(ids_arr)
        n_tune = max(1, int(round(len(ids_arr) * tune_frac)))
        tune_ids.update(int(x) for x in ids_arr[:n_tune])
        fit_ids.update(int(x) for x in ids_arr[n_tune:])
    return fit_ids, tune_ids


_, tune_ex_idx = stratified_ex_idx_split(train_pools, tune_frac=TUNE_FRAC, seed=SPLIT_SEED)
train_tune = [r for r in train_pools if int(r["ex_idx"]) in tune_ex_idx]
print(f"train_tune={len(train_tune):,} / {len(train_pools):,}")

## Core Analysis — train_tune grids

In [ ]:
def mean_ndcg_slice_a(pools: list[dict[str, Any]], *, alpha: float) -> float:
    vals: list[float] = []
    for row in pools:
        if int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = json.loads(row["retrieved_scores_json"])
        ex_idx = int(row["ex_idx"])
        blend = summary_blend_pool_scores(
            pool_apps, ret_sc, alpha=alpha,
            query_vec=train_query_vecs[ex_idx], summary_by_app=summary_by_app,
        )
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


train_grid = pd.DataFrame([
    {"alpha": alpha, "train_tune_NDCG_slice_a": mean_ndcg_slice_a(train_tune, alpha=alpha)}
    for alpha in BLEND_ALPHAS
]).sort_values("train_tune_NDCG_slice_a", ascending=False)
BEST_ALPHA = float(train_grid.iloc[0]["alpha"])

display(Markdown("### Train_tune grid — retr + summary (Slice A NDCG@10)"))
display(train_grid)
print(f"BEST_ALPHA={BEST_ALPHA}")
train_grid.to_csv(SPIKE_OUT / "v2b_train_tune_grid.csv", index=False)

In [ ]:
def mean_ndcg_slice_a_plus_d1(pools: list[dict[str, Any]], *, w_summary: float) -> float:
    vals: list[float] = []
    for row in pools:
        if int(row["n_eval_targets"]) < 2:
            continue
        positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
        if not positives:
            continue
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = json.loads(row["retrieved_scores_json"])
        ex_idx = int(row["ex_idx"])
        blend = d1_plus_summary_pool_scores(
            pool_apps, ret_sc, w_summary=w_summary,
            query_vec=train_query_vecs[ex_idx], summary_by_app=summary_by_app,
            pop_row=pop_row, app_to_row=app_to_row,
        )
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
        vals.append(ndcg_at_k(ranked, positives, K_FINAL, app_ids))
    return float(np.mean(vals)) if vals else float("nan")


plus_d1_grid = pd.DataFrame([
    {"w_summary": w, "train_tune_NDCG_slice_a": mean_ndcg_slice_a_plus_d1(train_tune, w_summary=w)}
    for w in PLUS_D1_WEIGHTS
]).sort_values("train_tune_NDCG_slice_a", ascending=False)
BEST_W_SUMMARY = float(plus_d1_grid.iloc[0]["w_summary"])

display(Markdown("### Train_tune grid — D1 + summary `_logpop_blend` (Slice A NDCG@10)"))
display(plus_d1_grid)
print(f"BEST_W_SUMMARY={BEST_W_SUMMARY}")
plus_d1_grid.to_csv(SPIKE_OUT / "v2b_plus_d1_train_tune_grid.csv", index=False)

## Evaluation — val face-off

In [ ]:
def per_example_ranking_row(
    row: dict[str, Any],
    *,
    method: str,
    score_fn: Callable[..., np.ndarray] | None = None,
    oracle: bool = False,
    catalog_pop: bool = False,
) -> dict[str, Any] | None:
    positives = set(int(x) for x in json.loads(row["validation_positive_app_ids_json"]))
    if not positives:
        return None
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    retrieved_rows = np.asarray([app_to_row[a] for a in pool_apps], dtype=np.int64)
    oracle_indices = _oracle_ranked_indices_from_retrieved(retrieved_rows, positives, app_ids)
    if oracle:
        ranked = oracle_indices[:K_FINAL]
    elif catalog_pop:
        s = np.asarray(pop_row, dtype=np.float64).copy()
        qrow = app_to_row.get(int(row["query_app_id"]))
        if qrow is not None:
            s[qrow] = -np.inf
        ranked = _rank_rows(s)[:K_FINAL]
    elif score_fn is None:
        ranked = pool_scores_to_ranked_indices(pool_apps, np.asarray(ret_sc), k_final=K_FINAL)
    else:
        blend = score_fn(pool_apps, ret_sc)
        ranked = pool_scores_to_ranked_indices(pool_apps, blend, k_final=K_FINAL)
    return {
        "method": method,
        "ex_idx": int(row["ex_idx"]),
        "slice_name": row["slice_name"],
        "n_eval_targets": int(row["n_eval_targets"]),
        "n_support_train": int(row.get("n_support_train", 0)),
        "query_app_id": int(row["query_app_id"]),
        "Hit@K": hit_rate_at_k(ranked, positives, K_FINAL, app_ids),
        "Precision@K": precision_at_k(ranked, positives, K_FINAL, app_ids),
        "Recall@K": recall_at_k(ranked, positives, K_FINAL, app_ids),
        "MAP@K": average_precision_at_k(ranked, positives, K_FINAL, app_ids),
        "NDCG@K": ndcg_at_k(ranked, positives, K_FINAL, app_ids),
        "MRR": mrr(ranked, positives, app_ids),
        "OracleHit@K": hit_rate_at_k(oracle_indices, positives, K_FINAL, app_ids),
        "OracleNDCG@K": ndcg_at_k(oracle_indices, positives, K_FINAL, app_ids),
    }


val_metric_rows: list[dict[str, Any]] = []
for row in val_pools:
    ex_idx = int(row["ex_idx"])
    qvec = val_query_vecs[ex_idx]
    qid = int(row["query_app_id"])
    configs: list[tuple[str, dict[str, Any]]] = [
        (POOL_METHOD, {}),
        ("two_tower_v1_heuristic_logpop_blend", {
            "score_fn": lambda pa, rs: score_logpop_blend(pa, rs, alpha=D1_ALPHA, pop_row=pop_row, app_to_row=app_to_row),
        }),
        ("popularity_train", {"catalog_pop": True}),
        (f"{POOL_METHOD}_oracle", {"oracle": True}),
        ("two_tower_v1_v2b_igdb_summary", {
            "score_fn": lambda pa, rs: summary_blend_pool_scores(
                pa, rs, alpha=BEST_ALPHA, query_vec=qvec, summary_by_app=summary_by_app,
            ),
        }),
        ("two_tower_v1_v2b_igdb_summary_logpop_blend", {
            "score_fn": lambda pa, rs: d1_plus_summary_pool_scores(
                pa, rs, w_summary=BEST_W_SUMMARY, query_vec=qvec, summary_by_app=summary_by_app,
                pop_row=pop_row, app_to_row=app_to_row,
            ),
        }),
        ("two_tower_v1_v2a_embed_query_logpop_blend", {
            "score_fn": lambda pa, rs: score_v2a_embed_query_logpop_blend(
                pa, rs, w_meta=DEFAULT_V2A_EMBED_W_META, query_app_id=qid,
                fields=V2A_GENRE_THEME_KW_FIELDS, pop_row=pop_row, app_to_row=app_to_row,
            ),
        }),
    ]
    for method, kwargs in configs:
        mrow = per_example_ranking_row(row, method=method, **kwargs)
        if mrow:
            val_metric_rows.append(mrow)

df_val = pd.DataFrame(val_metric_rows)
df_val.to_parquet(SPIKE_OUT / "v2b_val_per_example.parquet", index=False)

overall = _table_overall_ranking(df_val)
by_slice = _table_by_slice_for_metrics(df_val, metric_cols=RANKING_REPORT_METRIC_COLS, ranking=True)
by_support = _table_by_support_for_metrics(df_val, metric_cols=RANKING_REPORT_METRIC_COLS, ranking=True)
pop_table, pop_delta, _ = _table_popularity(
    df_ex_metrics=df_val, examples=examples_for_pers, app_ids=app_ids, pop_row=pop_row,
    enable_popularity_decile_diagnostics=True, metric_cols=RANKING_REPORT_METRIC_COLS,
)

overall.to_csv(SPIKE_OUT / "v2b_val_overall.csv", index=False)
by_slice.to_csv(SPIKE_OUT / "v2b_val_by_slice.csv", index=False)
by_support.to_csv(SPIKE_OUT / "v2b_val_by_support.csv", index=False)
pop_table.to_csv(SPIKE_OUT / "v2b_val_by_pop_decile.csv", index=False)
pop_delta.to_csv(SPIKE_OUT / "v2b_val_pop_delta.csv", index=False)

display(Markdown("### Val overall"))
display(overall.sort_values("NDCG@K", ascending=False))
display(Markdown("### Val by slice"))
display(by_slice.sort_values(["slice_name", "NDCG@K"], ascending=[True, False]))

## Personalization

In [ ]:
def frozen_pool_score(ex: dict[str, Any]) -> np.ndarray:
    row = val_pools_by_ex[int(ex["ex_idx"])]
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, sc in zip(pool_apps, ret_sc):
        full[int(app_to_row[int(app_id)])] = float(sc)
    return full.astype(np.float32)


def popularity_train_score(ex: dict[str, Any]) -> np.ndarray:
    s = np.asarray(pop_row, dtype=np.float64).copy()
    qrow = app_to_row.get(int(ex["query_app_id"]))
    if qrow is not None:
        s[qrow] = -np.inf
    return s.astype(np.float32)


def make_summary_pers_score_fn(*, logpop_blend: bool) -> Callable[[dict[str, Any]], np.ndarray]:
    def score(ex: dict[str, Any]) -> np.ndarray:
        row = val_pools_by_ex[int(ex["ex_idx"])]
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        qvec = val_query_vecs[int(ex["ex_idx"])]
        if logpop_blend:
            blend = d1_plus_summary_pool_scores(
                pool_apps, ret_sc, w_summary=BEST_W_SUMMARY, query_vec=qvec,
                summary_by_app=summary_by_app, pop_row=pop_row, app_to_row=app_to_row,
            )
        else:
            blend = summary_blend_pool_scores(
                pool_apps, ret_sc, alpha=BEST_ALPHA, query_vec=qvec, summary_by_app=summary_by_app,
            )
        full = np.full(len(app_ids), -np.inf, dtype=np.float64)
        for app_id, sc in zip(pool_apps, blend):
            full[int(app_to_row[int(app_id)])] = float(sc)
        return full.astype(np.float32)
    return score


def make_v2a_pers_score_fn() -> Callable[[dict[str, Any]], np.ndarray]:
    def score(ex: dict[str, Any]) -> np.ndarray:
        row = val_pools_by_ex[int(ex["ex_idx"])]
        pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
        ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
        blend = score_v2a_embed_query_logpop_blend(
            pool_apps, ret_sc, w_meta=DEFAULT_V2A_EMBED_W_META,
            query_app_id=int(ex["query_app_id"]), fields=V2A_GENRE_THEME_KW_FIELDS,
            pop_row=pop_row, app_to_row=app_to_row,
        )
        full = np.full(len(app_ids), -np.inf, dtype=np.float64)
        for app_id, sc in zip(pool_apps, blend):
            full[int(app_to_row[int(app_id)])] = float(sc)
        return full.astype(np.float32)
    return score


def d1_frozen_score(ex: dict[str, Any]) -> np.ndarray:
    row = val_pools_by_ex[int(ex["ex_idx"])]
    pool_apps = [int(x) for x in json.loads(row["retrieved_app_ids_json"])]
    ret_sc = [float(x) for x in json.loads(row["retrieved_scores_json"])]
    blend = score_logpop_blend(pool_apps, ret_sc, alpha=D1_ALPHA, pop_row=pop_row, app_to_row=app_to_row)
    full = np.full(len(app_ids), -np.inf, dtype=np.float64)
    for app_id, sc in zip(pool_apps, blend):
        full[int(app_to_row[int(app_id)])] = float(sc)
    return full.astype(np.float32)


pers_methods = {
    "popularity_train": popularity_train_score,
    POOL_METHOD: frozen_pool_score,
    "two_tower_v1_heuristic_logpop_blend": d1_frozen_score,
    "two_tower_v1_v2b_igdb_summary": make_summary_pers_score_fn(logpop_blend=False),
    "two_tower_v1_v2b_igdb_summary_logpop_blend": make_summary_pers_score_fn(logpop_blend=True),
    "two_tower_v1_v2a_embed_query_logpop_blend": make_v2a_pers_score_fn(),
}

personalization = _table_personalization(
    methods=pers_methods, examples=examples_for_pers, X=X_emb, app_ids=app_ids,
    pop_row=pop_row, k_personalization=K_PERSONALIZATION,
)
overall_pers = _append_personalization_metrics(overall.copy(), personalization, on_keys=["method"])
personalization.to_csv(SPIKE_OUT / "v2b_val_personalization.csv", index=False)
overall_pers.to_csv(SPIKE_OUT / "v2b_val_overall_with_personalization.csv", index=False)

display(Markdown("### Personalization"))
display(personalization.sort_values("PersonalizationGapVsPopularity@10", ascending=False))

# Key Findings

**Finding 1 (retr+summary — kill):**  
Best retr+summary train_tune: **α=0** (pure summary), Slice A **0.052**. Val `two_tower_v1_v2b_igdb_summary`: **0.032** / **0.036** overall / Slice A — far below D1.

**Finding 2 (`_logpop_blend` — defer vs v2a):**  
`two_tower_v1_v2b_igdb_summary_logpop_blend` (`w_summary=0.1`): val **0.093** / **0.072** overall / Slice A — **beats D1** (**0.093** / **0.068**) but **loses overall** to shipped v2a (**0.095** / **0.070**). Slice A marginally ahead of v2a.

**Finding 3:**  
Summary signal is **not redundant** with taxonomy v2a at small blend weight — D1+summary lifts Slice A similarly to D1+taxonomy, but neither combination dominates the other on all metrics.

**Unexpected Results:**  
α=0 (summary-only) wins train_tune over α>0, same pattern as v2a metadata — but val still collapses without D1 prior.

# Recommendation / Next Steps

**Recommended Action:**  
**Kill** summary-only rerank. **Defer** `two_tower_v1_v2b_igdb_summary_logpop_blend` — does not beat shipped v2a on overall NDCG. Next: **V2c** blend (v2a + v2b) if combining signals shows lift; optional head-to-head notebook for v2b vs v2a logpop_blend on pairwise NDCG.

**Risks:**  
Query embedding cost at inference (one USE forward per request). Summary text is publisher copy — may drift from player language.

**Follow-up Analyses:**  
V2c-query combined blend; correlation of summary vs taxonomy scores within pools; IGDB missing-summary slice report on full catalog join.

**Open Questions:**  
Does V2c (taxonomy + summary + D1) beat v2a alone? Is Slice A gain from v2b stable on a fresh val draw?